In [0]:
from pyspark.sql.functions import col

# --- Connect to the Bronze notebook's validated output ---
bronze_df = spark.table("final_project.bronze.loan_repayment_valid")

# Good and bad records (rejection_reason is set by the Bronze notebook)
valid_df = bronze_df.filter(col("rejection_reason") == "")
rejected_df = bronze_df.filter(col("rejection_reason") != "")

print("Valid records:", valid_df.count())
print("Rejected records:", rejected_df.count())

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date

# Clean and transform valid Bronze records

silver_df = (
    valid_df
    .withColumn("payment_id", trim(col("payment_id")))
    .withColumn("loan_id", trim(col("loan_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("loan_type", upper(trim(col("loan_type"))))
    .withColumn("branch_city", upper(trim(col("branch_city"))))
    .withColumn("payment_status", upper(trim(col("payment_status"))))
    .withColumn("payment_date", to_date(col("payment_date")))
    .withColumn("principal_amount", col("principal_amount").cast("double"))
    .withColumn("interest_amount", col("interest_amount").cast("double"))
    .withColumn("penalty_amount", col("penalty_amount").cast("double"))
    .withColumn(
        "total_payment",
        col("principal_amount") +
        col("interest_amount") +
        col("penalty_amount")
    )
    .drop("rejection_reason", "duplicate_count")
    .dropDuplicates(["payment_id"])
)

display(silver_df)

In [0]:
silver_df.printSchema()

In [0]:
print("Silver Record Count:", silver_df.count())

In [0]:
print(
    "Unique Payment IDs:",
    silver_df.select("payment_id").distinct().count()
)

In [0]:
display(silver_df.limit(10))

In [0]:
# Save Silver data

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Final_project.silver.loan_repayment")

In [0]:
print("Silver table created successfully")

In [0]:
# Read the saved Silver table

silver_table_df = spark.table("Final_project.silver.loan_repayment")

In [0]:
print("Silver Table Count:", silver_table_df.count())

In [0]:
display(silver_table_df)